In [ ]:
import torch
import numpy as np
from torch.nn import *
from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss, LigerRMSNorm, swiglu
from flash_attn import flash_attn_qkvpacked_func, flash_attn_kvpacked_func
from functools import reduce
from nn import Plot
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class Add(Sequential):
    def forward(s, _):
        return reduce(torch.add, (s(_)for s in s))
class T(Sequential):
    def forward(s, _):
        return super().forward(_.mT).mT
class F(Module):
    def __init__(s, _):
        super().__init__()
        s._ = _
    def forward(s, _):
        return s._(_)
class Attn(Linear):
    def __init__(s, i, h=0, G=1, g=1, r=(7,7), drop=0., alibi=1.):
        h = h or i
        super().__init__(i, G*(g+2)*h)
        s._ = Linear(G*g*h, i, 0)
        s.__dict__ |= locals()
    def forward(s, _):
        _ = super().forward(_)
        return s._((flash_attn_qkvpacked_func if s.g<2 else flash_attn_kvpacked_func)(
            *[_.unflatten(-1,(3,-1,s.h))]if s.g<2 else[_[...,:s.G*s.g*s.h].unflatten(-1,(-1,s.h)),_[...,s.G*s.g*s.h:].unflatten(-1,(2,-1,s.h))],
            dropout_p=s.drop,
            window_size=s.r,
            alibi_slopes=torch.arange(s.G*s.g*s.alibi, device='cuda'),
        ).flatten(-2))
class SwiGLU(Linear):
    def __init__(s, i, h=0):
        super().__init__(i, 2*(h or i))
        s._ = Linear(h or i, i, 0)
    def forward(s, _):
        return s._(swiglu.LigerSiLUMulFunction.apply(*super().forward(_).chunk(2,-1)))
def Cv(i, h=0, r=(7,7)):
    return Sequential(
        Linear(i, h or i, 0),
        T(Conv1d(h or i, h or i, sum(r)+1, padding=max(r), groups=h or i, bias=0)),
        *[F(lambda _:_[...,max(r[1]-r[0],0):min(r[1]-r[0],0)or None,:])] if r[1]!=r[0] else [],
        Linear(h or i, i, 0),
    )
def At(i, *_):
    return Sequential(
        LigerRMSNorm(i),
        Attn(i, *_)
    )
def Ff(i, *_):
    return Res(
        LigerRMSNorm(i),
        SwiGLU(i, *_),
    )
def U(i, h=0, s=2, S=0):
    return lambda*_:Res(
        Linear(i, h or i, 0),
        T(MaxPool1d(s)),
        *_,
        T(ConvTranspose1d(h or i, h or i, s, s, output_padding=max(S,0), groups=h or i, bias=0)),
        *[F(lambda _:_[...,S:,:]if S>0 else functional.pad(_[...,:S,:],(0,0,-S,0)))] if S else [],
        Linear(h or i, i, 0),
    )
torch._dynamo.config.cache_size_limit = 64
A = torch.from_numpy(np.memmap('../../hf/py65536.bin', mode='r'))
A = A.as_strided_((len(A)//8192-1,16384),(8192,1))

In [ ]:
i,H,h,G,g,a = 64,512,8,8,8,512
m = Sequential(
    Embedding(65536, i),
    Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
    U(i)(
        Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
        U(i)(
            Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
            U(i)(
                Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                U(i)(
                    Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                    U(i)(
                        Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                        U(i)(
                            Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                            U(i)(
                                Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                                U(i)(
                                    Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                                ),
                                Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                            ),
                            Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                        ),
                        Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                    ),
                    Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
                ),
                Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
            ),
            Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
        ),
        Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
    ),
    Add(Cv(i, H), At(i, h, G, g)), Ff(i, a),
).to('cuda',torch.bfloat16)
los = LigerFusedLinearCrossEntropyLoss()
try:
    m.load_state_dict(torch.load('m3.pk'))
except FileNotFoundError:
    m = torch.compile(m, dynamic=False)
    o = torch.optim.Adam(m.parameters(), 1/1024, weight_decay=1e-5)
    plot = Plot(2, 10, 10)
    for _ in range(100000):
        t = A[torch.randint(len(A),[16])].to('cuda',torch.long)
        x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(65536,[t.numel()//2],device='cuda'))
        y = m(x)
        l = los(getattr(m,'0').weight, y.flatten(0,1), t.ravel())
        (l*1024).backward()
        o.step()
        o.zero_grad()
        plot(l, lambda:l.neg().exp())
    torch.save(m._orig_mod.state_dict(), 'm3.pk')

In [ ]:
l = a = 0
with torch.no_grad():
    for _ in range(200):
        t = A[torch.randint(len(A),[3])].to('cuda',torch.long)
        x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(65536,[t.numel()//2],device='cuda'))
        y = m(x) @ m[0].weight.T
        l += functional.cross_entropy(y.mT, t).float()
        a += (y.argmax(2)==t).sum().float()/t.numel()
l/200, a/200

In [ ]:
class MyAdd(Add):
    _ = []
    def forward(s, _):
        _ = [s(_)for s in s]
        s._ += [_.norm()for _ in _],
        return reduce(torch.add,_)
class MyRes(Res):
    _ = []
    def forward(s, _):
        t = super().forward(_)
        s._ += [_.norm(),t.norm()],
        return _ + t
for _ in m.modules():
    if isinstance(_, Add):
        _.__class__ = MyAdd
    elif isinstance(_, Res):
        _.__class__ = MyRes
with torch.no_grad():
    m(x)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(torch.tensor(MyAdd._).to('cpu',torch.float))
plt.yscale('log')

In [ ]:
plt.plot(torch.tensor(MyRes._).to('cpu',torch.float))
plt.yscale('log')